# 2️⃣ Exploratory Data Analysis – Global Literacy & Education Trends

Univariate and bivariate analysis with Matplotlib, Seaborn, and Plotly.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from src.data_cleaning import get_cleaned_dataframes

df_literacy, df_illiteracy, df_gdp_schooling = get_cleaned_dataframes(ROOT / "data")
print("Shapes:", df_literacy.shape, df_illiteracy.shape, df_gdp_schooling.shape)

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

if "adult_literacy_rate" in df_literacy.columns:
    df_literacy["adult_literacy_rate"].dropna().hist(ax=axes[0,0], bins=30, edgecolor="black", alpha=0.7)
    axes[0,0].set_title("Distribution of Adult Literacy Rate")
    axes[0,0].set_xlabel("Adult Literacy (%)")

if "gdp_per_capita" in df_gdp_schooling.columns:
    df_gdp_schooling["gdp_per_capita"].dropna().hist(ax=axes[0,1], bins=40, edgecolor="black", alpha=0.7)
    axes[0,1].set_title("Distribution of GDP per Capita")
    axes[0,1].set_xlabel("GDP per capita")

if "avg_years_schooling" in df_gdp_schooling.columns:
    df_gdp_schooling["avg_years_schooling"].dropna().hist(ax=axes[1,0], bins=25, edgecolor="black", alpha=0.7)
    axes[1,0].set_title("Distribution of Avg Years of Schooling")
    axes[1,0].set_xlabel("Years")

if "illiteracy_pct" in df_illiteracy.columns:
    df_illiteracy["illiteracy_pct"].dropna().hist(ax=axes[1,1], bins=30, edgecolor="black", alpha=0.7)
    axes[1,1].set_title("Distribution of Illiteracy %")
    axes[1,1].set_xlabel("Illiteracy %")

plt.tight_layout()
plt.show()

## Bivariate: GDP vs Literacy

In [ ]:
merged = df_literacy.merge(df_gdp_schooling, on=["country", "year"], how="inner")
if "adult_literacy_rate" in merged.columns and "gdp_per_capita" in merged.columns:
    recent = merged[merged["year"] >= 2015].dropna(subset=["adult_literacy_rate", "gdp_per_capita"])
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=recent, x="gdp_per_capita", y="adult_literacy_rate", alpha=0.6)
    plt.title("Adult Literacy Rate vs GDP per Capita (2015+)")
    plt.xlabel("GDP per capita")
    plt.ylabel("Adult Literacy (%)")
    plt.xscale("log")
    plt.tight_layout()
    plt.show()

## Correlation Heatmap

In [ ]:
num_cols = ["adult_literacy_rate", "gdp_per_capita", "avg_years_schooling", "illiteracy_pct", "youth_literacy_avg"]
cols = [c for c in num_cols if c in merged.columns]
if cols:
    plt.figure(figsize=(10, 8))
    sns.heatmap(merged[cols].corr(), annot=True, fmt=".2f", cmap="RdYlGn", center=0)
    plt.title("Correlation Heatmap (Literacy, GDP, Schooling)")
    plt.tight_layout()
    plt.show()

## Literacy Trend Over Time (Global / Country)

In [ ]:
if "year" in df_literacy.columns and "adult_literacy_rate" in df_literacy.columns:
    global_avg = df_literacy.groupby("year")["adult_literacy_rate"].mean().reset_index()
    plt.figure(figsize=(10, 5))
    plt.plot(global_avg["year"], global_avg["adult_literacy_rate"], marker="o", markersize=4)
    plt.title("Global Average Adult Literacy Rate Over Time")
    plt.xlabel("Year")
    plt.ylabel("Adult Literacy (%)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Summary Insights
- Univariate: Literacy and schooling are right-skewed; GDP is highly right-skewed (log scale helps).
- Bivariate: Positive correlation between literacy and GDP; literacy and schooling are strongly related.
- Trends: Global adult literacy has improved over time; regional and country-level gaps remain.